# Public barefoot heel/toe fine-tuning — Colab GPU
Select **Runtime → Change runtime type → T4 GPU**.
This notebook does not use Roboflow training credits, API keys, or Google Drive mounts.
Upload only the official **Foot Keypoints Detection v5 COCO data ZIP**.
An optional checkpoint must match the project's known CMU pilot hash below.

Publisher: AI Training, Roboflow Universe. Declared license: Public Domain / CC0.
Source: https://universe.roboflow.com/ai-training-rvswa/foot-keypoints-detection/dataset/5
The publisher reports 820 unaugmented images (574/164/82); actual usable counts
are calculated after import. This is not a verified subject- or video-separated split.

Downloaded code is never executed. ZIP paths, file types, sizes and links are
checked before data is read. Images are decoded and newly encoded. These checks
reduce risk but are not a malware-free certification. No millimeter foot measurements,
metatarsal labels or shoe-fit outcomes are available, so this is not a sizing model.


In [ ]:
%pip install -q ultralytics==8.4.143 onnx onnxruntime pillow pyyaml


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU required. Select a T4 runtime; no silent CPU fallback.')
print('CUDA:', torch.cuda.is_available(), 'GPU:', torch.cuda.get_device_name(0))


In [ ]:
%%writefile prepare_public_barefoot.py
"""Read a Roboflow COCO data ZIP; never extract or execute downloaded code.

Only the published heel/big-toe/small-toe annotations are supervised. Ankle and
metatarsal slots remain masked. All output files are newly encoded data files.
"""
import argparse
from collections import defaultdict
from datetime import datetime, timezone
import hashlib
from io import BytesIO
import json
import math
from pathlib import Path, PurePosixPath
import re
import stat
import zipfile

SOURCE = "https://universe.roboflow.com/ai-training-rvswa/foot-keypoints-detection/dataset/5"
KEYPOINTS = ["heel", "ankle_unlabelled", "medial_ball_unlabelled", "lateral_ball_unlabelled", "big_toe", "small_toe"]
SPLITS = {"train": "train", "valid": "val", "val": "val", "test": "test"}
MAX_MEMBER = 32 * 1024 * 1024
MAX_TOTAL = 1024 * 1024 * 1024


def digest(data):
    return hashlib.sha256(data).hexdigest()


def safe_member(name):
    path = PurePosixPath(name)
    if (not name or path.is_absolute() or ".." in path.parts or "\\" in name
            or ":" in name or "\x00" in name or str(path) != name.rstrip("/")):
        raise ValueError(f"Unsafe ZIP path: {name!r}")
    return path


def inspect_archive(archive):
    """Reject executable payloads, ZIP bombs, links and unsafe paths before reading."""
    counts = defaultdict(int)
    total = 0
    seen = set()
    members = archive.infolist()
    if len(members) > 5000:
        raise ValueError("Too many ZIP members")
    for member in members:
        path = safe_member(member.filename)
        if member.filename in seen:
            raise ValueError("Duplicate ZIP member name")
        seen.add(member.filename)
        if stat.S_ISLNK(member.external_attr >> 16):
            raise ValueError("ZIP symlinks are not permitted")
        if member.flag_bits & 1:
            raise ValueError("Encrypted ZIP members are not permitted")
        if member.is_dir():
            continue
        suffix = path.suffix.lower()
        if suffix not in {".jpg", ".jpeg", ".png", ".json", ".txt"}:
            raise ValueError(f"Unexpected file type (not extracted): {suffix}")
        if suffix == ".txt" and path.name not in {"README.dataset.txt", "README.roboflow.txt"}:
            raise ValueError("Only dataset readme text is allowed in a COCO export")
        if member.file_size > MAX_MEMBER or member.file_size > max(1, member.compress_size) * 300:
            raise ValueError("Oversized or suspiciously compressed ZIP member")
        total += member.file_size
        if total > MAX_TOTAL:
            raise ValueError("ZIP exceeds the uncompressed data budget")
        counts[suffix] += 1
    return {"status": "data_only_structure_passed", "file_types": dict(counts),
            "uncompressed_bytes": total, "note": "Structural allowlist checks; not an antivirus certification"}


def semantic_name(name):
    cleaned = re.sub(r"[^a-z0-9]+", "_", str(name).lower()).strip("_")
    cleaned = re.sub(r"^(left|right)_", "", cleaned)
    aliases = {"heel": "heel", "big_toe": "big_toe", "bigtoe": "big_toe",
               "small_toe": "small_toe", "smalltoe": "small_toe", "little_toe": "small_toe"}
    return aliases.get(cleaned)


def source_points(annotation, category, width, height):
    names = category.get("keypoints", [])
    mapping = [semantic_name(name) for name in names]
    if len(names) != 3 or set(mapping) != {"heel", "big_toe", "small_toe"}:
        raise ValueError(f"Unrecognised keypoint names; no guessed mapping: {names}")
    raw = annotation.get("keypoints", [])
    if len(raw) == 3 and all(isinstance(point, list) and len(point) == 3 for point in raw):
        triples = raw
    elif len(raw) == 9 and all(isinstance(value, (int, float)) for value in raw):
        triples = [raw[index:index + 3] for index in range(0, 9, 3)]
    else:
        raise ValueError("Expected exactly three labelled source keypoints")
    points = {}
    for name, point in zip(mapping, triples):
        x, y, visibility = point
        if not all(isinstance(value, (int, float)) and math.isfinite(value) for value in point):
            raise ValueError("Nonfinite keypoint")
        if visibility not in (0, 1, 2):
            raise ValueError("Invalid keypoint visibility")
        if visibility and not (0 <= x <= width and 0 <= y <= height):
            raise ValueError("Keypoint outside image")
        points[name] = point if visibility else [0, 0, 0]
    if not points["heel"][2] or not points["big_toe"][2]:
        return None
    heel, toe = points["heel"], points["big_toe"]
    if math.hypot(toe[0] - heel[0], toe[1] - heel[1]) < 10:
        return None
    return [heel, [0, 0, 0], [0, 0, 0], [0, 0, 0], toe, points["small_toe"]]


def label_line(annotation, category, width, height):
    if annotation.get("iscrowd", 0):
        return None
    points = source_points(annotation, category, width, height)
    if points is None:
        return None
    box = annotation.get("bbox", [])
    if len(box) != 4 or not all(isinstance(value, (int, float)) and math.isfinite(value) for value in box):
        raise ValueError("Invalid source bbox")
    x, y, box_width, box_height = box
    if (box_width <= 0 or box_height <= 0 or x < 0 or y < 0
            or x + box_width > width + 1 or y + box_height > height + 1):
        raise ValueError("Source bbox outside image")
    box_width, box_height = min(box_width, width - x), min(box_height, height - y)
    values = [0, (x + box_width / 2) / width, (y + box_height / 2) / height,
              box_width / width, box_height / height]
    for px, py, visibility in points:
        values.extend([px / width, py / height, visibility] if visibility else [0, 0, 0])
    return " ".join(str(value) if index == 0 else f"{value:.6f}" for index, value in enumerate(values))


def prepare(zip_path, output):
    from PIL import Image
    output = Path(output).resolve()
    if output.exists() and any(output.iterdir()):
        raise ValueError("Use an empty output directory; existing data is preserved")
    zip_path = Path(zip_path)
    if zip_path.stat().st_size > MAX_TOTAL:
        raise ValueError("ZIP exceeds the compressed data budget")
    archive_bytes = zip_path.read_bytes()
    records, excluded, categories_record, annotation_sources = [], [], {}, []
    hashes = set()
    with zipfile.ZipFile(BytesIO(archive_bytes)) as archive:
        security = inspect_archive(archive)
        annotations = [name for name in archive.namelist() if name.endswith("_annotations.coco.json")]
        configs = {}
        for annotation_file in annotations:
            parent = PurePosixPath(annotation_file).parent
            split = SPLITS.get(parent.name)
            if split is None or split in configs:
                raise ValueError("Expected one COCO annotation file per split")
            annotation_bytes = archive.read(annotation_file)
            document = json.loads(annotation_bytes, parse_constant=lambda value: (_ for _ in ()).throw(ValueError(value)))
            configs[split] = (parent, document)
            annotation_sources.append({"file": annotation_file, "sha256": digest(annotation_bytes)})
        if set(configs) != {"train", "val", "test"}:
            raise ValueError("COCO export must contain train, validation and test")
        output.mkdir(parents=True, exist_ok=True)
        for split in ("test", "val", "train"):
            parent, document = configs[split]
            categories = {category["id"]: category for category in document["categories"]}
            categories_record[split] = list(categories.values())
            grouped = defaultdict(list)
            for annotation in document["annotations"]:
                grouped[annotation["image_id"]].append(annotation)
            (output / "images" / split).mkdir(parents=True)
            (output / "labels" / split).mkdir(parents=True)
            seen_ids = set()
            for info in document["images"]:
                if type(info["id"]) is not int or info["id"] < 0:
                    raise ValueError("Image IDs must be nonnegative integers")
                if info["id"] in seen_ids:
                    raise ValueError("Duplicate COCO image ID in split")
                seen_ids.add(info["id"])
                source_file = str(parent / safe_member(info["file_name"]))
                data = archive.read(source_file)
                with Image.open(BytesIO(data)) as image:
                    image.verify()
                with Image.open(BytesIO(data)) as image:
                    if image.format not in {"JPEG", "PNG"} or image.width * image.height > 32_000_000:
                        raise ValueError("Unsupported image content or dimensions")
                    image = image.convert("RGB")
                    width, height = image.size
                    if (width, height) != (info["width"], info["height"]):
                        raise ValueError("Image and annotation dimensions disagree")
                    pixel_hash = digest(image.tobytes() + f"{width}x{height}".encode())
                    if pixel_hash in hashes:
                        excluded.append({"file": source_file, "reason": "exact_pixel_duplicate; evaluation takes precedence"})
                        continue
                    lines = [line for annotation in grouped[info["id"]]
                             if (line := label_line(annotation, categories[annotation["category_id"]], width, height))]
                    if not lines:
                        excluded.append({"file": source_file, "reason": "no usable heel-toe annotation; not treated as a negative"})
                        continue
                    hashes.add(pixel_hash)
                    stem = f"rf5_{split}_{info['id']}"
                    prepared = output / "images" / split / f"{stem}.jpg"
                    image.save(prepared, quality=95)
                (output / "labels" / split / f"{stem}.txt").write_text("\n".join(lines) + "\n", encoding="utf-8")
                records.append({"source_identity": source_file, "image_id": info["id"], "split": split,
                                "file": f"images/{split}/{stem}.jpg", "source_file": source_file,
                                "original_sha256": digest(data), "pixel_sha256": pixel_hash,
                                "prepared_sha256": digest(prepared.read_bytes()), "foot_count": len(lines)})
    counts = {split: sum(record["split"] == split for record in records) for split in ("train", "val", "test")}
    feet = {split: sum(record["foot_count"] for record in records if record["split"] == split) for split in counts}
    report = {"source": SOURCE, "created_utc": datetime.now(timezone.utc).isoformat(),
              "archive_sha256": digest(archive_bytes), "archive_security_check": security,
              "declared_license": "Public Domain / CC0 as declared by AI Training on Roboflow Universe",
              "annotation_sources": annotation_sources, "source_categories": categories_record,
              "keypoints": KEYPOINTS, "unlabelled_keypoints": [1, 2, 3], "metric_ground_truth": False,
              "image_counts": counts, "foot_counts": feet, "records": records, "excluded": excluded,
              "split_policy": "Publisher image split; exact pixel duplicates removed with test > val > train priority. No subject/video-exclusive guarantee."}
    (output / "manifest.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    if any(count < 1 for count in counts.values()):
        raise ValueError("A split has no usable images; do not train")
    yaml = (f"path: {output.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\n"
            "kpt_shape: [6, 3]\nflip_idx: [0, 1, 2, 3, 4, 5]\nnames:\n  0: foot\n")
    (output / "data.yaml").write_text(yaml, encoding="utf-8")
    print(json.dumps({"images": counts, "feet": feet, "excluded": len(excluded), "archive_check": security}), flush=True)
    return report


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--zip", required=True)
    parser.add_argument("--output", default="training/public_barefoot_rf5")
    args = parser.parse_args()
    prepare(args.zip, args.output)


In [ ]:
%%writefile train_public_footnet.py
"""Train/evaluate a public-data pilot without replacing the app's existing model."""
import argparse
from datetime import datetime, timezone
import hashlib
import json
import math
from pathlib import Path
import time


def audit_dataset(data_path):
    import yaml
    config = yaml.safe_load(data_path.read_text(encoding="utf-8"))
    root = Path(config["path"]).resolve()
    manifest = json.loads((root / "manifest.json").read_text(encoding="utf-8"))
    if manifest.get("metric_ground_truth") is not False:
        raise ValueError("This pipeline expects nonmetric public pose data")
    if config.get("kpt_shape") != [6, 3] or config.get("names") != {0: "foot"}:
        raise ValueError("Expected the web engine's single-class, six-slot pose layout")
    seen = set()
    hash_splits = {}
    actual_counts = {split: 0 for split in ("train", "val", "test")}
    actual_feet = dict(actual_counts)
    for record in manifest["records"]:
        if record["split"] not in actual_counts:
            raise ValueError("Invalid dataset split")
        identity = record.get("source_identity", (record.get("coco_split"), record["image_id"]))
        if isinstance(identity, list):
            identity = tuple(identity)
        if identity in seen:
            raise ValueError("Source image duplicated across dataset records")
        seen.add(identity)
        image_hash = record.get('original_sha256')
        if image_hash and image_hash in hash_splits and hash_splits[image_hash] != record['split']:
            raise ValueError('Identical image contents appear in different splits')
        if image_hash:
            hash_splits[image_hash] = record['split']
        if "coco_split" in record and record["split"] == "train" and record["coco_split"] != "train2017":
            raise ValueError("Training must use official train images only")
        image = (root / record["file"]).resolve()
        if not image.is_relative_to(root):
            raise ValueError("Dataset image path escapes dataset directory")
        label = root / "labels" / record["split"] / (image.stem + ".txt")
        if not image.exists() or not label.exists():
            raise FileNotFoundError(image)
        if record.get("prepared_sha256") and hashlib.sha256(image.read_bytes()).hexdigest() != record["prepared_sha256"]:
            raise ValueError("Prepared dataset image hash has changed")
        actual_counts[record["split"]] += 1
        foot_count = 0
        for line in label.read_text().splitlines():
            values = [float(v) for v in line.split()]
            if (len(values) != 23 or not all(math.isfinite(v) for v in values)
                    or values[0] != 0 or any(not 0 <= v <= 1 for v in values[1:5])
                    or values[3] <= 0 or values[4] <= 0):
                raise ValueError(f"Invalid YOLO label: {label}")
            for index in range(6):
                x, y, visibility = values[5 + index * 3:8 + index * 3]
                if visibility not in (0, 1, 2) or not 0 <= x <= 1 or not 0 <= y <= 1:
                    raise ValueError(f"Invalid keypoint: {label}")
                if index in (2, 3) and visibility != 0:
                    raise ValueError("Unobserved metatarsals must not be fabricated")
            foot_count += 1
        if foot_count != record["foot_count"]:
            raise ValueError("Manifest foot count does not match labels")
        actual_feet[record["split"]] += foot_count
    if actual_counts != manifest["image_counts"] or actual_feet != manifest["foot_counts"]:
        raise ValueError("Manifest counts do not match actual dataset files")
    if any(count < 1 for count in actual_counts.values()):
        raise ValueError("All three splits are required")
    return manifest


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", required=True)
    parser.add_argument("--weights", default="training/yolov8n-pose.pt")
    parser.add_argument("--epochs", type=int, default=20)
    parser.add_argument("--batch", type=int, default=16)
    parser.add_argument("--imgsz", type=int, default=256)
    parser.add_argument("--device", default="cpu")
    parser.add_argument("--threads", type=int, default=4)
    parser.add_argument("--freeze", type=int, default=10)
    parser.add_argument("--project", default="training/runs/public")
    parser.add_argument("--name", default="cmu_pilot")
    parser.add_argument("--baseline", default="")
    args = parser.parse_args()
    import torch
    from ultralytics import YOLO
    data = Path(args.data).resolve()
    manifest = audit_dataset(data)
    torch.set_num_threads(args.threads)
    started = time.time()
    model = YOLO(args.weights)
    # Freeze pretrained backbone for an affordable CPU pilot; --freeze 0 on GPU.
    results = model.train(data=str(data), epochs=args.epochs, imgsz=args.imgsz,
                          batch=args.batch, device=args.device, workers=0,
                          project=str(Path(args.project).resolve()), name=args.name,
                          exist_ok=False, optimizer="AdamW", lr0=0.001, lrf=0.01,
                          freeze=args.freeze, seed=42, deterministic=True,
                          amp=args.device != "cpu", plots=True, patience=15,
                          mosaic=0.2, close_mosaic=5, degrees=25, fliplr=0.5,
                          translate=0.1, scale=0.3)
    run = Path(model.trainer.save_dir)
    best = run / "weights" / "best.pt"
    candidate = YOLO(str(best))
    validation = candidate.val(data=str(data), split="val", imgsz=args.imgsz, device=args.device,
                               workers=0, project=str(run), name="validation", plots=False)
    test = candidate.val(data=str(data), split="test", imgsz=args.imgsz, device=args.device,
                         workers=0, project=str(run), name="heldout_test", plots=True)
    baseline_metrics = None
    if args.baseline:
        baseline = YOLO(args.baseline)
        baseline_test = baseline.val(data=str(data), split="test", imgsz=args.imgsz,
                                     device=args.device, workers=0, project=str(run),
                                     name="baseline_heldout", plots=False)
        baseline_metrics = {key: float(value) for key, value in baseline_test.results_dict.items()}
    export = candidate.export(format="onnx", imgsz=args.imgsz, batch=1, half=False,
                              dynamic=False, simplify=False, opset=17, device="cpu")
    import numpy as np
    import onnxruntime as ort
    session = ort.InferenceSession(str(export), providers=["CPUExecutionProvider"])
    input_name = session.get_inputs()[0].name
    output = session.run(None, {input_name: np.zeros((1, 3, args.imgsz, args.imgsz), dtype=np.float32)})[0]
    if output.shape[0] != 1 or output.shape[1] != 23 or not np.isfinite(output).all():
        raise RuntimeError(f"Unexpected web output: {output.shape}")
    report = {"created_utc": datetime.now(timezone.utc).isoformat(), "stage": "public_data_pilot",
              "checkpoint": str(best), "onnx": str(export), "onnx_sha256": hashlib.sha256(Path(export).read_bytes()).hexdigest(),
              "onnx_input": session.get_inputs()[0].shape, "onnx_output": list(output.shape),
              "dataset_source": manifest["source"], "image_counts": manifest["image_counts"],
              "foot_counts": manifest["foot_counts"], "keypoints": manifest["keypoints"],
              "actual_epochs": len(model.trainer.csv.read_text().splitlines()) - 1,
              "training_config": vars(args), "elapsed_seconds": round(time.time() - started),
              "validation": {key: float(value) for key, value in validation.results_dict.items()},
              "heldout_test": {key: float(value) for key, value in test.results_dict.items()},
              "previous_synthetic_baseline_test": baseline_metrics,
              "metric_sizing_validated": False, "production_ready": False,
              "split_policy": manifest.get("split_policy"),
              "limitations": ["No foot length/width in mm or product fit labels",
                              "K2/K3 have no ground truth; do not use them for sizing",
                              "Image split, not a verified subject-exclusive split",
                              "Training images may differ from real mobile scans",
                              "Image rights vary by source; retain dataset attribution",
                              "Real-phone tracking and metric sizing require independent validation"]}
    (run / "training_report.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
    print("PUBLIC TRAINING COMPLETE", flush=True)
    print(json.dumps(report, indent=2), flush=True)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile validate_public_onnx.py
"""Check exported ONNX numerical agreement on real held-out image crops."""
import argparse
import json
from pathlib import Path

import numpy as np
from PIL import Image
import torch
import onnxruntime as ort
from ultralytics import YOLO

parser = argparse.ArgumentParser()
parser.add_argument('--run', default='training/runs/public/cmu_pilot')
parser.add_argument('--images', default='training/public_cmu_pilot/images/test')
args = parser.parse_args()
run = Path(args.run)
torch.set_num_threads(4)
model = YOLO(str(run / 'weights/best.pt')).model.eval()
session = ort.InferenceSession(str(run / 'weights/best.onnx'), providers=['CPUExecutionProvider'])
input_name = session.get_inputs()[0].name
size = session.get_inputs()[0].shape[2]
records = []
for path in sorted(Path(args.images).glob('*.jpg'))[:3]:
    with Image.open(path) as image:
        image = image.convert('RGB')
        width, height = image.size
        # Preserve the whole visible image with a grey letterbox, as the current
        # web inference geometry does. Do not silently crop portrait feet away.
        scale = min(size / width, size / height)
        draw_width, draw_height = round(width * scale), round(height * scale)
        boxed = Image.new('RGB', (size, size), (114, 114, 114))
        resized = image.resize((draw_width, draw_height), Image.Resampling.BILINEAR)
        boxed.paste(resized, ((size - draw_width) // 2, (size - draw_height) // 2))
        pixels = np.asarray(boxed, dtype=np.float32).transpose(2, 0, 1)[None] / 255.0
    with torch.no_grad():
        torch_output = model(torch.from_numpy(pixels))[0].numpy()
    onnx_output = session.run(None, {input_name: pixels})[0]
    if not np.isfinite(onnx_output).all() or onnx_output.shape != torch_output.shape:
        raise RuntimeError('Invalid real-image ONNX output')
    difference = np.abs(torch_output - onnx_output)
    if difference.max() > 0.01:
        raise RuntimeError(f'ONNX export disagreement: {difference.max()}')
    records.append({'image': path.name, 'shape': list(onnx_output.shape),
                    'max_absolute_difference': float(difference.max()),
                    'mean_absolute_difference': float(difference.mean())})
if len(records) != 3:
    raise RuntimeError('Three held-out images are required')
report = {'status': 'passed', 'tolerance': 0.01, 'preprocessing': 'full-image RGB letterbox, grey 114', 'records': records,
          'scope': 'Numerical export equivalence only; not mobile performance or sizing accuracy'}
(run / 'onnx_validation.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))


## Data input
Upload the COCO ZIP. Optionally upload the trusted project checkpoint `best.pt`
from the previous CMU pilot; its SHA-256 must match the embedded hash.
Do not upload private phone captures, credentials, vendor scripts or arbitrary weights.
Files remain in the temporary Colab runtime; download results before it disconnects.


In [ ]:
from google.colab import files
from pathlib import Path
import hashlib

uploaded = files.upload()
zips = [name for name in uploaded if Path(name).suffix.lower() == '.zip']
weights = [name for name in uploaded if Path(name).suffix.lower() == '.pt']
if len(zips) != 1 or len(weights) > 1 or len(uploaded) != len(zips) + len(weights):
    raise ValueError('Upload one public COCO ZIP and optionally the trusted best.pt only.')
dataset_zip = Path(zips[0]).resolve()
initial_weights = 'yolov8n-pose.pt'  # official Ultralytics pretrained weights
baseline_weights = ''
if weights:
    checkpoint = Path(weights[0]).resolve()
    if hashlib.sha256(checkpoint.read_bytes()).hexdigest() != 'd08ccc832e2183ccbdb721a3e5f76573bcb6f44b9f1d26d0b516d7c44ec5404b':
        raise ValueError('Unrecognised checkpoint; do not load or execute arbitrary weights.')
    initial_weights = str(checkpoint)
    baseline_weights = str(checkpoint)
print('Dataset ZIP SHA-256:', hashlib.sha256(dataset_zip.read_bytes()).hexdigest())


In [ ]:
from prepare_public_barefoot import prepare
from train_public_footnet import audit_dataset

data_dir = Path('/content/public_barefoot_rf5')
if (data_dir / 'data.yaml').exists():
    manifest = audit_dataset(data_dir / 'data.yaml')
    if manifest['archive_sha256'] != hashlib.sha256(dataset_zip.read_bytes()).hexdigest():
        raise ValueError('Runtime contains a different dataset. Use a clean runtime.')
else:
    manifest = prepare(dataset_zip, data_dir)
    audit_dataset(data_dir / 'data.yaml')
print('Actual images:', manifest['image_counts'])
print('Actual labelled feet:', manifest['foot_counts'])
print('Split limitation:', manifest['split_policy'])


In [ ]:
import subprocess, sys

# Forward logs to notebook output instead of losing subprocess stdout.
def run_stream(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
    if code:
        raise subprocess.CalledProcessError(code, command)

command = [sys.executable, '-u', 'train_public_footnet.py', '--data', str(data_dir / 'data.yaml'),
           '--weights', initial_weights, '--epochs', '60', '--batch', '32', '--imgsz', '256',
           '--device', '0', '--freeze', '0', '--project', '/content/runs', '--name', 'barefoot_rf5_gpu']
if baseline_weights:
    command += ['--baseline', baseline_weights]
run_stream(command)


In [ ]:
import json, shutil

reports = sorted(Path('/content/runs').glob('barefoot_rf5_gpu*/training_report.json'),
                 key=lambda path: path.stat().st_mtime)
if not reports:
    raise RuntimeError('Training has not completed; no release candidate is ready.')
report_path = reports[-1]
run_stream([sys.executable, '-u', 'validate_public_onnx.py', '--run', str(report_path.parent),
            '--images', str(data_dir / 'images/test')])
report = json.loads(report_path.read_text())
print('Trained UTC:', report['created_utc'])
print('Actual epochs:', report['actual_epochs'])
print('Held-out pose metrics (not sizing accuracy):', report['heldout_test'])
print('Candidate ONNX SHA-256:', report['onnx_sha256'])
shutil.copy(data_dir / 'manifest.json', report_path.parent / 'dataset_manifest.json')
result_zip = shutil.make_archive('/content/barefoot_public_results', 'zip', report_path.parent)
files.download(result_zip)
